In [2]:
from __future__ import annotations

import os
import sys
import json
import random
from pathlib import Path
from typing import Dict, Any, List

import torch
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.datasets import fetch_20newsgroups
from transformers import AutoTokenizer, BertTokenizer


# ============================================================
# ROOTS
# ============================================================

ROOT = Path.cwd()

VENV_EXPECTED = ROOT / ".venv_a100" / "bin" / "python"

HF_MODEL_DIR = ROOT / "hf_models"

DATA_ROOT = ROOT / "real_text_datasets"
RAW_DIR = DATA_ROOT / "raw"
TOKENIZED_DIR = DATA_ROOT / "tokenized_pt"
REPORT_DIR = DATA_ROOT / "reports"

HF_DATASETS_CACHE = DATA_ROOT / "hf_datasets_cache"

for p in [RAW_DIR, TOKENIZED_DIR, REPORT_DIR, HF_DATASETS_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE)
os.environ["HF_HOME"] = str(ROOT / "hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ============================================================
# CONFIG
# ============================================================

SEED = 42
MAX_LEN = 128

# None = full datasets.
# For smoke test, set e.g. 2000 / 500.
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

DATASETS = [
    {
        "name": "ag_news",
        "source": "hf",
        "hf_candidates": [
            "fancyzhx/ag_news",
            "ag_news",
        ],
        "text_col": "text",
        "label_col": "label",
        "num_labels": 4,
        "task_type": "topic_classification",
    },
    {
        "name": "imdb",
        "source": "hf",
        "hf_candidates": [
            "stanfordnlp/imdb",
            "imdb",
        ],
        "text_col": "text",
        "label_col": "label",
        "num_labels": 2,
        "task_type": "sentiment_classification",
    },
    {
        "name": "20newsgroups",
        "source": "sklearn",
        "text_col": "text",
        "label_col": "label",
        "num_labels": 20,
        "task_type": "topic_classification",
    },
]

MODELS = [
    {
        "model_name": "distilbert_base_uncased",
        "model_path": HF_MODEL_DIR / "distilbert_base_uncased",
        "tokenizer_loader": "auto",
    },
    {
        "model_name": "bert_tiny",
        "model_path": HF_MODEL_DIR / "bert_tiny",
        "tokenizer_loader": "bert_forced",
    },
    {
        "model_name": "bert_mini",
        "model_path": HF_MODEL_DIR / "bert_mini",
        "tokenizer_loader": "bert_forced",
    },
    {
        "model_name": "electra_small_discriminator",
        "model_path": HF_MODEL_DIR / "electra_small_discriminator",
        "tokenizer_loader": "auto",
    },
]


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_env() -> None:
    print("=" * 100)
    print("ENVIRONMENT")
    print("=" * 100)
    print("ROOT:", ROOT)
    print("Python executable:", sys.executable)
    print("Expected venv python:", VENV_EXPECTED)
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
    print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
    print("HF_HOME:", os.environ.get("HF_HOME"))

    if ".venv_a100" not in sys.executable:
        print("\nWARNING: this does not look like .venv_a100 Python.")

    print("=" * 100)


def limit_split(ds: Dataset, max_samples: int | None, seed: int) -> Dataset:
    if max_samples is None:
        return ds

    n = min(len(ds), max_samples)
    return ds.shuffle(seed=seed).select(range(n))


def normalize_columns(ds: DatasetDict, text_col: str, label_col: str) -> DatasetDict:
    """
    Ensures columns are named text / label if needed.
    """

    out = {}

    for split_name, split_ds in ds.items():
        cols = split_ds.column_names

        rename_map = {}

        if text_col not in cols:
            # common alternatives
            for alt in ["content", "sentence", "review", "description"]:
                if alt in cols:
                    rename_map[alt] = text_col
                    break

        if label_col not in cols:
            for alt in ["labels", "target", "category"]:
                if alt in cols:
                    rename_map[alt] = label_col
                    break

        if rename_map:
            split_ds = split_ds.rename_columns(rename_map)

        keep_cols = [c for c in [text_col, label_col] if c in split_ds.column_names]
        extra_cols = [c for c in split_ds.column_names if c not in keep_cols]

        if extra_cols:
            split_ds = split_ds.remove_columns(extra_cols)

        out[split_name] = split_ds

    return DatasetDict(out)


def save_dataset_as_jsonl(ds: Dataset, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for row in ds:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_dataset_as_csv(ds: Dataset, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ds.to_pandas().to_csv(path, index=False)


def load_tokenizer(model_spec: Dict[str, Any]):
    model_path = model_spec["model_path"]

    if model_spec["tokenizer_loader"] == "bert_forced":
        return BertTokenizer.from_pretrained(
            model_path,
            local_files_only=True,
        )

    return AutoTokenizer.from_pretrained(
        model_path,
        local_files_only=True,
        use_fast=False,
    )


def tokenize_split(
    ds: Dataset,
    tokenizer,
    text_col: str,
    label_col: str,
    max_len: int,
) -> Dict[str, torch.Tensor]:
    input_ids = []
    attention_mask = []
    labels = []

    for row in tqdm(ds, desc="Tokenizing", leave=False):
        text = str(row[text_col])
        label = int(row[label_col])

        enc = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )

        input_ids.append(enc["input_ids"].squeeze(0))
        attention_mask.append(enc["attention_mask"].squeeze(0))
        labels.append(label)

    return {
        "input_ids": torch.stack(input_ids).long(),
        "attention_mask": torch.stack(attention_mask).long(),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


# ============================================================
# DATASET DOWNLOADERS
# ============================================================

def load_hf_with_candidates(candidates: List[str]) -> tuple[DatasetDict, str]:
    last_error = None

    for name in candidates:
        print(f"\nTrying HF dataset: {name}")

        try:
            ds = load_dataset(
                name,
                cache_dir=str(HF_DATASETS_CACHE),
                trust_remote_code=False,
            )

            print("Loaded HF dataset:", name)
            return ds, name

        except Exception as e:
            last_error = e
            print("FAILED:", name)
            print("ERROR:", repr(e))

    raise RuntimeError(f"All HF candidates failed. Last error: {repr(last_error)}")


def prepare_hf_dataset(cfg: Dict[str, Any]) -> tuple[DatasetDict, str]:
    print("\n" + "=" * 100)
    print("Downloading HF dataset candidates:", cfg["hf_candidates"])
    print("=" * 100)

    ds, used_name = load_hf_with_candidates(cfg["hf_candidates"])

    if "train" not in ds:
        raise RuntimeError(f"{used_name}: no train split")

    if "test" not in ds:
        split = ds["train"].train_test_split(test_size=0.2, seed=SEED)
        ds = DatasetDict({
            "train": split["train"],
            "test": split["test"],
        })

    ds = DatasetDict({
        "train": limit_split(ds["train"], MAX_TRAIN_SAMPLES, SEED),
        "test": limit_split(ds["test"], MAX_TEST_SAMPLES, SEED),
    })

    ds = normalize_columns(
        ds,
        text_col=cfg["text_col"],
        label_col=cfg["label_col"],
    )

    return ds, used_name


def prepare_20newsgroups_dataset(cfg: Dict[str, Any]) -> tuple[DatasetDict, str]:
    print("\n" + "=" * 100)
    print("Downloading sklearn dataset: 20 Newsgroups")
    print("=" * 100)

    train = fetch_20newsgroups(
        subset="train",
        remove=("headers", "footers", "quotes"),
    )

    test = fetch_20newsgroups(
        subset="test",
        remove=("headers", "footers", "quotes"),
    )

    train_df = pd.DataFrame({
        "text": train.data,
        "label": train.target.astype(int),
    })

    test_df = pd.DataFrame({
        "text": test.data,
        "label": test.target.astype(int),
    })

    train_ds = Dataset.from_pandas(train_df, preserve_index=False)
    test_ds = Dataset.from_pandas(test_df, preserve_index=False)

    train_ds = limit_split(train_ds, MAX_TRAIN_SAMPLES, SEED)
    test_ds = limit_split(test_ds, MAX_TEST_SAMPLES, SEED)

    return DatasetDict({
        "train": train_ds,
        "test": test_ds,
    }), "sklearn_20newsgroups"


def prepare_dataset(cfg: Dict[str, Any]) -> tuple[DatasetDict, str]:
    if cfg["source"] == "hf":
        return prepare_hf_dataset(cfg)

    if cfg["source"] == "sklearn":
        return prepare_20newsgroups_dataset(cfg)

    raise ValueError(f"Unknown source: {cfg['source']}")


# ============================================================
# SAVE RAW DATASET
# ============================================================

def save_raw_dataset(cfg: Dict[str, Any], ds: DatasetDict, used_source: str) -> Dict[str, Any]:
    name = cfg["name"]
    out_dir = RAW_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\nSaving raw dataset:", name)
    print("Raw dir:", out_dir)

    ds.save_to_disk(str(out_dir / "datasetdict"))

    save_dataset_as_jsonl(ds["train"], out_dir / "train.jsonl")
    save_dataset_as_jsonl(ds["test"], out_dir / "test.jsonl")

    save_dataset_as_csv(ds["train"], out_dir / "train.csv")
    save_dataset_as_csv(ds["test"], out_dir / "test.csv")

    info = {
        "name": name,
        "source": cfg["source"],
        "used_source": used_source,
        "task_type": cfg["task_type"],
        "text_col": cfg["text_col"],
        "label_col": cfg["label_col"],
        "num_labels": cfg["num_labels"],
        "train_size": len(ds["train"]),
        "test_size": len(ds["test"]),
        "raw_dir": str(out_dir),
        "splits": list(ds.keys()),
        "train_columns": ds["train"].column_names,
        "test_columns": ds["test"].column_names,
    }

    with open(out_dir / "dataset_info.json", "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4, ensure_ascii=False)

    return info


# ============================================================
# TOKENIZATION
# ============================================================

def tokenize_for_model(
    dataset_cfg: Dict[str, Any],
    ds: DatasetDict,
    model_spec: Dict[str, Any],
) -> Dict[str, Any]:
    dataset_name = dataset_cfg["name"]
    model_name = model_spec["model_name"]

    print("\n" + "-" * 100)
    print("Tokenizing dataset/model:")
    print("dataset:", dataset_name)
    print("model:", model_name)
    print("-" * 100)

    out_dir = TOKENIZED_DIR / dataset_name / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    train_pt = out_dir / "train.pt"
    test_pt = out_dir / "test.pt"
    info_path = out_dir / "tokenized_info.json"

    if train_pt.exists() and test_pt.exists() and info_path.exists():
        print("Already tokenized, skipping:", out_dir)
        with open(info_path, "r", encoding="utf-8") as f:
            return json.load(f)

    tokenizer = load_tokenizer(model_spec)

    train_tensors = tokenize_split(
        ds["train"],
        tokenizer,
        text_col=dataset_cfg["text_col"],
        label_col=dataset_cfg["label_col"],
        max_len=MAX_LEN,
    )

    test_tensors = tokenize_split(
        ds["test"],
        tokenizer,
        text_col=dataset_cfg["text_col"],
        label_col=dataset_cfg["label_col"],
        max_len=MAX_LEN,
    )

    torch.save(train_tensors, train_pt)
    torch.save(test_tensors, test_pt)

    info = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "model_path": str(model_spec["model_path"]),
        "tokenizer_loader": model_spec["tokenizer_loader"],
        "max_len": MAX_LEN,
        "num_labels": dataset_cfg["num_labels"],
        "task_type": dataset_cfg["task_type"],
        "train_size": int(train_tensors["labels"].shape[0]),
        "test_size": int(test_tensors["labels"].shape[0]),
        "train_pt": str(train_pt),
        "test_pt": str(test_pt),
        "train_shape_input_ids": list(train_tensors["input_ids"].shape),
        "test_shape_input_ids": list(test_tensors["input_ids"].shape),
        "train_labels_unique": sorted([int(x) for x in train_tensors["labels"].unique().tolist()]),
        "test_labels_unique": sorted([int(x) for x in test_tensors["labels"].unique().tolist()]),
    }

    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4, ensure_ascii=False)

    print("Saved tokenized:")
    print(" ", train_pt)
    print(" ", test_pt)

    return info


def smoke_check_tokenized(info: Dict[str, Any]) -> Dict[str, Any]:
    train_pt = Path(info["train_pt"])
    test_pt = Path(info["test_pt"])

    train = torch.load(train_pt, map_location="cpu")
    test = torch.load(test_pt, map_location="cpu")

    result = {
        "dataset_name": info["dataset_name"],
        "model_name": info["model_name"],
        "train_ok": False,
        "test_ok": False,
        "train_n": None,
        "test_n": None,
        "seq_len": None,
        "num_labels": info["num_labels"],
        "error": None,
    }

    try:
        for data in [train, test]:
            assert "input_ids" in data
            assert "attention_mask" in data
            assert "labels" in data
            assert data["input_ids"].ndim == 2
            assert data["attention_mask"].shape == data["input_ids"].shape
            assert data["labels"].ndim == 1
            assert data["labels"].shape[0] == data["input_ids"].shape[0]

        result["train_ok"] = True
        result["test_ok"] = True
        result["train_n"] = int(train["labels"].shape[0])
        result["test_n"] = int(test["labels"].shape[0])
        result["seq_len"] = int(train["input_ids"].shape[1])

    except Exception as e:
        result["error"] = repr(e)

    return result


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    set_seed(SEED)
    print_env()

    all_dataset_info = []
    all_tokenized_info = []
    all_smoke = []

    for dataset_cfg in DATASETS:
        try:
            ds, used_source = prepare_dataset(dataset_cfg)

            dataset_info = save_raw_dataset(dataset_cfg, ds, used_source)
            all_dataset_info.append(dataset_info)

            print("\nDataset info:")
            print(json.dumps(dataset_info, indent=4, ensure_ascii=False))

            for model_spec in MODELS:
                model_path = model_spec["model_path"]

                if not model_path.exists():
                    print("\nWARNING: local model path missing, skipping:")
                    print(" ", model_path)
                    continue

                tokenized_info = tokenize_for_model(
                    dataset_cfg=dataset_cfg,
                    ds=ds,
                    model_spec=model_spec,
                )

                all_tokenized_info.append(tokenized_info)

                smoke = smoke_check_tokenized(tokenized_info)
                all_smoke.append(smoke)

                print("Smoke:", smoke)

        except Exception as e:
            print("\nDATASET FAILED:", dataset_cfg["name"])
            print("ERROR:", repr(e))

            all_dataset_info.append({
                "name": dataset_cfg["name"],
                "source": dataset_cfg["source"],
                "status": "failed",
                "error": repr(e),
            })

    dataset_report_path = REPORT_DIR / "real_datasets_report.json"
    tokenized_report_path = REPORT_DIR / "real_tokenized_report.json"
    smoke_report_path = REPORT_DIR / "real_tokenized_smoke_report.json"

    with open(dataset_report_path, "w", encoding="utf-8") as f:
        json.dump(all_dataset_info, f, indent=4, ensure_ascii=False)

    with open(tokenized_report_path, "w", encoding="utf-8") as f:
        json.dump(all_tokenized_info, f, indent=4, ensure_ascii=False)

    with open(smoke_report_path, "w", encoding="utf-8") as f:
        json.dump(all_smoke, f, indent=4, ensure_ascii=False)

    pd.DataFrame(all_dataset_info).to_csv(REPORT_DIR / "real_datasets_report.csv", index=False)
    pd.DataFrame(all_tokenized_info).to_csv(REPORT_DIR / "real_tokenized_report.csv", index=False)
    pd.DataFrame(all_smoke).to_csv(REPORT_DIR / "real_tokenized_smoke_report.csv", index=False)

    print("\n" + "=" * 100)
    print("DONE")
    print("=" * 100)
    print("Raw datasets:", RAW_DIR)
    print("Tokenized PT:", TOKENIZED_DIR)
    print("Reports:", REPORT_DIR)

    print("\nSummary:")
    for x in all_dataset_info:
        if x.get("status") == "failed":
            print(f"{x['name']:15s} | FAILED | {x['error']}")
        else:
            print(
                f"{x['name']:15s} | "
                f"source={x['used_source']} | "
                f"train={x['train_size']} | "
                f"test={x['test_size']} | "
                f"labels={x['num_labels']} | "
                f"type={x['task_type']}"
            )


if __name__ == "__main__":
    main()

ENVIRONMENT
ROOT: /home/user/fractional_unlearning
Python executable: /home/user/fractional_unlearning/.venv_a100/bin/python
Expected venv python: /home/user/fractional_unlearning/.venv_a100/bin/python
torch: 2.6.0+cu124
cuda available: True
gpu: NVIDIA A100-PCIE-40GB
HF_DATASETS_CACHE: /home/user/fractional_unlearning/real_text_datasets/hf_datasets_cache
HF_HOME: /home/user/fractional_unlearning/hf_cache


Trying HF dataset: fancyzhx/ag_news


Generating train split: 100%|█| 120000/120000 [00:00<00:00, 927150.23 examples/s
Generating test split: 100%|█████| 7600/7600 [00:00<00:00, 785992.46 examples/s]


Loaded HF dataset: fancyzhx/ag_news

Saving raw dataset: ag_news
Raw dir: /home/user/fractional_unlearning/real_text_datasets/raw/ag_news


Saving the dataset (1/1 shards): 100%|█| 120000/120000 [00:00<00:00, 1869786.02 
Saving the dataset (1/1 shards): 100%|█| 7600/7600 [00:00<00:00, 750128.50 examp



Dataset info:
{
    "name": "ag_news",
    "source": "hf",
    "used_source": "fancyzhx/ag_news",
    "task_type": "topic_classification",
    "text_col": "text",
    "label_col": "label",
    "num_labels": 4,
    "train_size": 120000,
    "test_size": 7600,
    "raw_dir": "/home/user/fractional_unlearning/real_text_datasets/raw/ag_news",
    "splits": [
        "train",
        "test"
    ],
    "train_columns": [
        "text",
        "label"
    ],
    "test_columns": [
        "text",
        "label"
    ]
}

----------------------------------------------------------------------------------------------------
Tokenizing dataset/model:
dataset: ag_news
model: distilbert_base_uncased
----------------------------------------------------------------------------------------------------


...


Dataset info:
{
    "name": "20newsgroups",
    "source": "sklearn",
    "used_source": "sklearn_20newsgroups",
    "task_type": "topic_classification",
    "text_col": "text",
    "label_col": "label",
    "num_labels": 20,
    "train_size": 11314,
    "test_size": 7532,
    "raw_dir": "/home/user/fractional_unlearning/real_text_datasets/raw/20newsgroups",
    "splits": [
        "train",
        "test"
    ],
    "train_columns": [
        "text",
        "label"
    ],
    "test_columns": [
        "text",
        "label"
    ]
}

----------------------------------------------------------------------------------------------------
Tokenizing dataset/model:
dataset: 20newsgroups
model: distilbert_base_uncased
----------------------------------------------------------------------------------------------------


Saved tokenized:
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/distilbert_base_uncased/train.pt
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/distilbert_base_uncased/test.pt
Smoke: {'dataset_name': '20newsgroups', 'model_name': 'distilbert_base_uncased', 'train_ok': True, 'test_ok': True, 'train_n': 11314, 'test_n': 7532, 'seq_len': 128, 'num_labels': 20, 'error': None}

----------------------------------------------------------------------------------------------------
Tokenizing dataset/model:
dataset: 20newsgroups
model: bert_tiny
----------------------------------------------------------------------------------------------------


Saved tokenized:
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/bert_tiny/train.pt
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/bert_tiny/test.pt
Smoke: {'dataset_name': '20newsgroups', 'model_name': 'bert_tiny', 'train_ok': True, 'test_ok': True, 'train_n': 11314, 'test_n': 7532, 'seq_len': 128, 'num_labels': 20, 'error': None}

----------------------------------------------------------------------------------------------------
Tokenizing dataset/model:
dataset: 20newsgroups
model: bert_mini
----------------------------------------------------------------------------------------------------


Saved tokenized:
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/bert_mini/train.pt
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/bert_mini/test.pt
Smoke: {'dataset_name': '20newsgroups', 'model_name': 'bert_mini', 'train_ok': True, 'test_ok': True, 'train_n': 11314, 'test_n': 7532, 'seq_len': 128, 'num_labels': 20, 'error': None}

----------------------------------------------------------------------------------------------------
Tokenizing dataset/model:
dataset: 20newsgroups
model: electra_small_discriminator
----------------------------------------------------------------------------------------------------


Saved tokenized:
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/electra_small_discriminator/train.pt
  /home/user/fractional_unlearning/real_text_datasets/tokenized_pt/20newsgroups/electra_small_discriminator/test.pt
Smoke: {'dataset_name': '20newsgroups', 'model_name': 'electra_small_discriminator', 'train_ok': True, 'test_ok': True, 'train_n': 11314, 'test_n': 7532, 'seq_len': 128, 'num_labels': 20, 'error': None}

DONE
Raw datasets: /home/user/fractional_unlearning/real_text_datasets/raw
Tokenized PT: /home/user/fractional_unlearning/real_text_datasets/tokenized_pt
Reports: /home/user/fractional_unlearning/real_text_datasets/reports

Summary:
ag_news         | source=fancyzhx/ag_news | train=120000 | test=7600 | labels=4 | type=topic_classification
imdb            | source=stanfordnlp/imdb | train=25000 | test=25000 | labels=2 | type=sentiment_classification
20newsgroups    | source=sklearn_20newsgroups | train=11314 | test=7532 | labels=20 | type=to